# Atlanta Mobility CUDA routing benchmark
In Colab choose **Runtime → Change runtime type → T4 GPU**. Download the fresh JSON receipts before this ephemeral session ends.

In [ ]:
!git clone https://github.com/triasha72/atlanta-mobility-resilience-digital-twin.git || (cd atlanta-mobility-resilience-digital-twin && git pull --ff-only)
%cd /content/atlanta-mobility-resilience-digital-twin
%pip install -q -e '.[dev,gnn]'
!nvidia-smi || echo 'No NVIDIA GPU is attached. Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun.'

In [ ]:
# Install the RAPIDS build matched to this Colab runtime.
!git clone --depth 1 https://github.com/rapidsai/rapidsai-csp-utils.git || git -C rapidsai-csp-utils pull --ff-only
!python rapidsai-csp-utils/colab/pip-install.py
!python -c "import cugraph; print('cuGraph', cugraph.__version__)"

In [ ]:
!PYTHONPATH=src python scripts/probe_cuda_runtime.py --output reports/colab_gpu_probe.json
!cat reports/colab_gpu_probe.json

In [ ]:
# Rebuild ignored public inputs and the tract graph in this fresh runtime.
!PYTHONPATH=src python scripts/materialize_acs_origins.py
!PYTHONPATH=src python scripts/materialize_osm_destinations.py
!PYTHONPATH=src python scripts/benchmark_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cpu_benchmark.json
!cat reports/colab_cpu_benchmark.json

In [ ]:
# Equivalent directed, minimum-parallel-edge cuGraph routing over the same OD set.
!PYTHONPATH=src python scripts/benchmark_cugraph_routing.py --config configs/v1_atlanta_tract_study.yaml --output reports/colab_cugraph_benchmark.json
!cat reports/colab_cugraph_benchmark.json

A valid GPU receipt requires `cugraph_available: true` and a completed `colab_cugraph_benchmark.json`. Do not characterize the result as a speedup unless its graph size and OD count match the CPU report.